# Etapa 2 · Prueba tu agente Paulina

**Autor:** Paulina. **Grupo/integrantes:** Grupo 6.

Adaptación de `a2a-test.ipynb` del taller. Primero consultamos la API de LangGraph; después repetimos una consulta por A2A. El agente y sus herramientas permanecen sin cambios.

Este archivo está **preparado para ejecutarse**. Las respuestas válidas para la entrega son las que obtengas al correr las celdas; guarda con **Ctrl+S y no limpies las salidas**. Ejecutar todo realizará consultas al modelo configurado en el servidor.

> **Integración Grupo 6:** en el repositorio compartido este perfil corre en el puerto **2027**
> (Rodrigo en 2025, Christopher en 2026). El notebook de la Etapa 3 está en
> [`../../notebooks/a2a-test.ipynb`](../../notebooks/a2a-test.ipynb).

## Antes de ejecutar

1. En una terminal PowerShell, sitúate en la carpeta `agente-paulina` y ejecuta:

```powershell
$env:PYTHONUTF8 = "1"
.\.venv\Scripts\langgraph.exe dev --port 2027 --no-browser
```

Deja esa terminal abierta. `PYTHONUTF8` evita problemas con caracteres de los logs en Windows. La clave debe estar en el `.env` del servidor; no se copia al notebook.

2. Abre este notebook y selecciona como kernel el intérprete **agente-paulina/.venv/Scripts/python.exe**. Si no aparece, en VS Code utiliza «Seleccionar kernel → Entornos de Python → Seleccionar intérprete» y busca esa ruta.
3. Ejecuta las celdas de arriba abajo con **Shift+Enter**. Conserva la salida de descubrimiento, las tres pruebas, la tarjeta y la Task A2A.

In [1]:
import json
import os
import sys
import uuid
from pathlib import Path
import httpx
from langgraph_sdk import get_client

LANGGRAPH_URL = os.environ.get("A2A_URL", "http://127.0.0.1:2027")
A2A_HOST = "127.0.0.1"
A2A_PORT = int(os.environ.get("A2A_PORT", "2027"))
A2A_URL = os.environ.get("A2A_URL", f"http://{A2A_HOST}:{A2A_PORT}")
GRAPH_ID = "agente_paulina"
TIMEOUT = 180

# Solo verifica la ruta del kernel; no lee ni muestra claves.
carpeta_actual = Path.cwd().resolve()
PROYECTO = next((p for p in [carpeta_actual, *carpeta_actual.parents]
                 if (p / "langgraph.json").is_file() and (p / "src/agente_paulina").is_dir()), None)
if PROYECTO is None:
    raise RuntimeError("Abre el notebook dentro de agente-paulina/notebooks.")
if Path(sys.prefix).resolve() != (PROYECTO / ".venv").resolve():
    raise RuntimeError(f"Selecciona el kernel {PROYECTO / '.venv/Scripts/python.exe'}. Actual: {sys.executable}")

# No enviar claves de LangSmith/Google desde el notebook al servidor local.
lg = get_client(url=LANGGRAPH_URL, api_key=None, timeout=TIMEOUT)
evidencias = {}

def mostrar(valor):
    print(json.dumps(valor, indent=2, ensure_ascii=False, default=str))

print("Kernel:", sys.executable)
print("Servidor:", LANGGRAPH_URL)
print("Grafo esperado:", GRAPH_ID)

Kernel: C:\Users\Admin\Desktop\PYTHON\Agents\grupo-6-taller-a2a\agente-paulina\.venv\Scripts\python.exe
Servidor: http://127.0.0.1:2027
Grafo esperado: agente_paulina


## 1. Encontrar el Id de tu agente

El `graph_id` es el nombre registrado. El `assistant_id` es el UUID asignado por el servidor. Esta celda lista los asistentes y guarda la evidencia del descubrimiento.

In [2]:
try:
    asistentes = await lg.assistants.search(limit=100, response_format="array")
except httpx.ConnectError as exc:
    raise RuntimeError("No se pudo conectar. Inicia langgraph dev --port 2027 desde agente-paulina y revisa su terminal.") from exc

for asistente in asistentes:
    mostrar({k: asistente.get(k) for k in ("name", "graph_id", "assistant_id")})
if not asistentes:
    raise RuntimeError("El servidor no devolvió asistentes. Revisa langgraph.json y la terminal.")

{
  "name": "agente_paulina",
  "graph_id": "agente_paulina",
  "assistant_id": "80c74e98-5251-5ce6-8713-2b1dd5835c0b"
}


In [3]:
# Equivale a copiar el UUID correspondiente a tu grafo, sin fijar el de otra instalación.
candidatos = [a for a in asistentes if a.get("graph_id") == GRAPH_ID]
if len(candidatos) != 1:
    raise RuntimeError(f"Se esperaba un asistente para {GRAPH_ID}; encontrados: {len(candidatos)}. Revisa el listado.")
ASSISTANT_ID = candidatos[0]["assistant_id"]
uuid.UUID(ASSISTANT_ID)
evidencias["descubrimiento"] = {"graph_id": GRAPH_ID, "assistant_id": ASSISTANT_ID}
print("ASSISTANT_ID =", ASSISTANT_ID)

ASSISTANT_ID = 80c74e98-5251-5ce6-8713-2b1dd5835c0b


## 2. Comunicación directa con el agente · funciones auxiliares

`crear_entrada` construye el mensaje. `extraer_texto` recupera la respuesta final del estado, que puede incluir mensajes y resultados de herramientas. `consultar_langgraph` hace la consulta. Con `thread_id=None` es independiente; con el mismo hilo conserva contexto.

In [4]:
OUTPUT_KEY = None

def crear_entrada(texto):
    return {"messages": [{"role": "human", "content": texto}]}

def extraer_texto(estado):
    if not isinstance(estado, dict):
        raise ValueError("La salida no es un estado de grafo.")
    if estado.get("__interrupt__"):
        raise RuntimeError("El grafo quedó pausado; revisa el interrupt.")
    if OUTPUT_KEY:
        contenido = estado[OUTPUT_KEY]
    else:
        respuesta = next((m for m in reversed(estado.get("messages", []))
                          if m.get("type", m.get("role")) in ("ai", "assistant")), None)
        if respuesta is None:
            raise ValueError("No hay respuesta ai/assistant. Revisa la terminal del servidor.")
        contenido = respuesta.get("content", "")
    if isinstance(contenido, list):
        contenido = "\n".join(b if isinstance(b, str) else b.get("text", "")
                              for b in contenido if isinstance(b, (str, dict)))
    if not isinstance(contenido, str) or not contenido.strip():
        raise ValueError("El agente no devolvió texto final.")
    return contenido

def herramientas_usadas(estado):
    return [t.get("name", "") for m in estado.get("messages", [])
            for t in m.get("tool_calls", [])]

async def consultar_langgraph(texto, thread_id=None):
    estado = await lg.runs.wait(thread_id, ASSISTANT_ID, input=crear_entrada(texto))
    return extraer_texto(estado)

## 3. Pregunta 1 · Personalidad

Debe presentarse como Paulina, especialista en estrategia y valor de negocio de IA y datos, con tono directo, cercano y orientado a resultados. Puedes cambiar `PREGUNTA` y volver a ejecutar esta celda. Cada ejecución inicia una consulta independiente.

In [5]:
PREGUNTA = "Preséntate y cuéntame cómo me puedes ayudar."
estado_personalidad = await lg.runs.wait(None, ASSISTANT_ID, input=crear_entrada(PREGUNTA))
respuesta_personalidad = extraer_texto(estado_personalidad)
print("PREGUNTA:", PREGUNTA)
print("\nRESPUESTA:\n", respuesta_personalidad)
evidencias["personalidad"] = {"pregunta": PREGUNTA, "respuesta": respuesta_personalidad}

PREGUNTA: Preséntate y cuéntame cómo me puedes ayudar.

RESPUESTA:
 Soy Paulina, especialista en estrategia y valor de negocio de IA y datos. Ayudo a convertir ideas técnicas en decisiones: qué problema resuelven, cómo se mide el éxito (mermas, tiempos, quiebres, errores, horas, satisfacción) y qué valor o retorno se espera.

También armo roadmaps por fases partiendo por quick wins, con entregable y KPI de salida en cada una.

No diseño arquitectura ni calculo costos de desarrollo; para eso derivo a Rodrigo y Christopher, aunque uso sus costos como dato para evaluar retorno.

¿Tienes una iniciativa en mente? Cuéntame el problema y los usuarios afectados y vemos el caso de negocio.


## 4. Pregunta 2 · Conocimientos y skills

Pedimos evaluar primero el impacto operacional en los quiebres de stock y luego el retorno del caso ficticio. Revisa que consulte `consultar_mis_apuntes`, use `evaluar_impacto_kpi` y `calcular_retorno`. Debe tratar el caso como ficticio y declarar los supuestos.

**Referencia del caso acordado:** línea base 8 % de quiebres; meta 4 % (mejora de 4 puntos porcentuales, -50 %). Costo del piloto 9.000.000 CLP; beneficio 1.200.000 CLP/mes; a 12 meses el beneficio acumulado es 14.400.000 CLP, el ROI es 60,00 % y el payback es 7,5 meses. Es una referencia para contrastar, no una respuesta insertada como evidencia.

In [6]:
PREGUNTA_CONOCIMIENTOS = (
    "Usa el caso ficticio de tus apuntes para evaluar primero el impacto operacional y luego el retorno. "
    "Con evaluar_impacto_kpi mide los quiebres de stock: línea base 8 %, valor esperado 4 %, unidad %, "
    "mejor_si menor y sin valor_unitario_clp; la monetización se hace después con calcular_retorno. "
    "Después usa calcular_retorno con el costo del piloto (9.000.000 CLP) y el beneficio mensual del caso "
    "(1.200.000 CLP) a 12 meses. Consulta tus apuntes. "
    "Muestra la variación absoluta y porcentual del KPI, el beneficio acumulado, el ROI, los meses de recuperación "
    "y una conclusión breve. Declara los supuestos y recuerda que el caso es ficticio."
)
estado_conocimientos = await lg.runs.wait(None, ASSISTANT_ID, input=crear_entrada(PREGUNTA_CONOCIMIENTOS))
respuesta_conocimientos = extraer_texto(estado_conocimientos)
usadas = herramientas_usadas(estado_conocimientos)
print("PREGUNTA:", PREGUNTA_CONOCIMIENTOS)
print("\nRESPUESTA:\n", respuesta_conocimientos)
print("\nHERRAMIENTAS OBSERVADAS:", usadas)
evidencias["conocimientos"] = {"pregunta": PREGUNTA_CONOCIMIENTOS, "respuesta": respuesta_conocimientos, "herramientas": usadas}
if not {"consultar_mis_apuntes", "evaluar_impacto_kpi", "calcular_retorno"}.issubset(usadas):
    print("REVISAR: no aparecen todas las herramientas esperadas. Registra lo observado antes de ajustar el agente.")

PREGUNTA: Usa el caso ficticio de tus apuntes para evaluar primero el impacto operacional y luego el retorno. Con evaluar_impacto_kpi mide los quiebres de stock: línea base 8 %, valor esperado 4 %, unidad %, mejor_si menor y sin valor_unitario_clp; la monetización se hace después con calcular_retorno. Después usa calcular_retorno con el costo del piloto (9.000.000 CLP) y el beneficio mensual del caso (1.200.000 CLP) a 12 meses. Consulta tus apuntes. Muestra la variación absoluta y porcentual del KPI, el beneficio acumulado, el ROI, los meses de recuperación y una conclusión breve. Declara los supuestos y recuerda que el caso es ficticio.

RESPUESTA:
 **Conclusión:** el piloto mejora el KPI a la mitad y se paga dentro del horizonte: recuperación en 7,5 meses y ROI de 60 % a 12 meses. Caso ficticio de taller, no experiencia real.

**1) Impacto operacional (evaluar_impacto_kpi)**
- KPI: quiebres de stock en productos de alta rotación
- Línea base 8 % → esperado 4 %: variación absoluta **-

## 5. Pregunta 3 · Memoria

Creamos un hilo una sola vez y hacemos dos llamadas con el mismo `thread_id`. Al repetir esta primera celda crearás otra conversación. Para seguir la conversación actual, vuelve a ejecutar solo la celda siguiente, sin crear otro hilo.

In [7]:
thread = await lg.threads.create()
thread_id = thread["thread_id"]
OBJETIVO = "Evaluar un piloto de pronóstico de demanda para reducir quiebres de stock en 10 tiendas de retail."
MENSAJE_OBJETIVO = f"Mi objetivo para este proyecto es: {OBJETIVO} Guárdalo como contexto para esta conversación."
respuesta_objetivo = await consultar_langgraph(MENSAJE_OBJETIVO, thread_id=thread_id)
print("THREAD_ID:", thread_id)
print("MENSAJE:", MENSAJE_OBJETIVO)
print("\nRESPUESTA:\n", respuesta_objetivo)

THREAD_ID: 01a16a26-17ec-703c-85c9-5279383b6161
MENSAJE: Mi objetivo para este proyecto es: Evaluar un piloto de pronóstico de demanda para reducir quiebres de stock en 10 tiendas de retail. Guárdalo como contexto para esta conversación.

RESPUESTA:
 Anotado: piloto de pronóstico de demanda en 10 tiendas de retail, foco en reducir quiebres de stock.

Para armarte un caso de negocio y un roadmap necesito tres datos: línea base de quiebres (por ejemplo, % de SKU-día sin stock hoy), meta esperada tras el piloto, y horizonte/usuarios afectados. ¿Los tienes?


In [8]:
PREGUNTA_MEMORIA = "¿Qué objetivo te acabo de contar?"
respuesta_memoria = await consultar_langgraph(PREGUNTA_MEMORIA, thread_id=thread_id)
print("MISMO THREAD_ID:", thread_id)
print("PREGUNTA:", PREGUNTA_MEMORIA)
print("\nRESPUESTA:\n", respuesta_memoria)
evidencias["memoria"] = {"thread_id": thread_id, "objetivo": OBJETIVO, "primera_respuesta": respuesta_objetivo, "pregunta": PREGUNTA_MEMORIA, "respuesta": respuesta_memoria}

MISMO THREAD_ID: 01a16a26-17ec-703c-85c9-5279383b6161
PREGUNTA: ¿Qué objetivo te acabo de contar?

RESPUESTA:
 Me contaste: evaluar un piloto de pronóstico de demanda para reducir quiebres de stock en 10 tiendas de retail. Es el contexto con el que voy a trabajar en esta conversación.


## 6. Vamos a Agent2Agent · descargar la tarjeta

Hasta ahora usamos la API de LangGraph. Ahora usamos el protocolo A2A. El resolver pide la tarjeta incluyendo el `assistant_id`; sin ese parámetro el servidor puede rechazar la petición.

La tarjeta describe capacidades y dirección del agente. Fijamos `protocol_version = "0.3.0"` en la interfaz JSONRPC porque `langgraph-api` 0.16 incluye campos `kind` que el protobuf de `a2a-sdk` 1.2.2 no reconoce al parsear la Task. Es la misma compatibilidad que usa el notebook de Etapa 3.

In [9]:
from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.types import Role, SendMessageRequest, TaskState
from a2a.helpers import new_text_message, get_message_text, get_artifact_text
from google.protobuf.json_format import MessageToDict

# Permite repetir esta celda sin dejar clientes abiertos.
if "cliente_a2a" in globals():
    await cliente_a2a.close()
if "http_a2a" in globals() and not http_a2a.is_closed:
    await http_a2a.aclose()

http_a2a = httpx.AsyncClient(timeout=TIMEOUT, trust_env=False)
resolver = A2ACardResolver(http_a2a, A2A_URL)
card_remota = await resolver.get_agent_card(
    http_kwargs={"params": {"assistant_id": ASSISTANT_ID}}
)
for interfaz in card_remota.supported_interfaces:
    if interfaz.protocol_binding == "JSONRPC":
        # langgraph-api 0.16 incluye "kind" en la Task; el protobuf de a2a-sdk 1.2.2
        # no lo reconoce. 0.3.0 usa el formato compatible del transporte JSONRPC.
        interfaz.protocol_version = "0.3.0"

print("AGENT CARD DESCARGADA:")
mostrar(MessageToDict(card_remota))
evidencias["tarjeta"] = MessageToDict(card_remota)

cliente_a2a = await create_client(
    card_remota,
    client_config=ClientConfig(
        httpx_client=http_a2a, streaming=False,
        supported_protocol_bindings=["JSONRPC"],
    ),
)
print("Cliente A2A preparado. Mantener abierto hasta enviar el mensaje.")

AGENT CARD DESCARGADA:
{
  "name": "agente_paulina",
  "description": "Especialista en estrategia y valor de negocio de IA y datos. Define el caso de negocio (problema, KPIs y beneficio esperado), evalúa el impacto de una iniciativa con el KPI que corresponda (mermas, tiempos de proceso, quiebres de stock, errores, horas ahorradas, satisfacción) y, cuando hay costo y beneficio monetario confiables, también su retorno. Propone un roadmap por fases partiendo por quick wins. No diseña arquitectura ni calcula costos de desarrollo; usa los aportes de otros especialistas como datos.",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:2027/a2a/80c74e98-5251-5ce6-8713-2b1dd5835c0b",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "0.3.0"
    }
  ],
  "version": "0.16.6",
  "capabilities": {
    "streaming": true,
    "pushNotifications": false,
    "extensions": [
      {
        "uri": "https://langchain.com/a2a/extensions/history-scope/v1",
        "description": "

## 7. Repetir la consulta por A2A · Message → Task → Artifact

Repetimos la consulta del caso de los apuntes, ahora por A2A. El mensaje tiene su propio identificador y un `contextId` UUID. La Task identifica el trabajo; sus artifacts contienen la salida. Mostramos el resultado estructurado completo y el texto.

Este hilo A2A es independiente de la prueba de memoria anterior. Al repetir esta celda conserva su UUID. El envío puede esperar mientras el modelo consulta las herramientas; si falla, revisa también la terminal del servidor.

In [10]:
if http_a2a.is_closed:
    raise RuntimeError("El cliente está cerrado. Reejecuta la celda de tarjeta antes de enviar.")
if "context_id_a2a" not in globals():
    context_id_a2a = str(uuid.uuid4())

PREGUNTA_A2A = PREGUNTA_CONOCIMIENTOS
mensaje_a2a = new_text_message(PREGUNTA_A2A, role=Role.ROLE_USER, context_id=context_id_a2a)
solicitud_a2a = SendMessageRequest(message=mensaje_a2a)
print("MENSAJE A2A:")
mostrar(MessageToDict(mensaje_a2a))

textos_a2a = []
tareas_a2a = []
async for evento in cliente_a2a.send_message(solicitud_a2a):
    if evento.HasField("task"):
        tarea = evento.task
        tareas_a2a.append(MessageToDict(tarea))
        print("\nTASK DEVUELTA:")
        mostrar(MessageToDict(tarea))
        print("ESTADO:", TaskState.Name(tarea.status.state))
        if tarea.status.state != TaskState.TASK_STATE_COMPLETED:
            raise RuntimeError(f"La Task {tarea.id} no terminó correctamente. Revisa su estado y la terminal.")
        textos_a2a.extend(get_artifact_text(a) for a in tarea.artifacts)
        if not any(textos_a2a) and tarea.status.HasField("message"):
            textos_a2a.append(get_message_text(tarea.status.message))
    elif evento.HasField("message"):
        print("\nRESPUESTA TIPO MESSAGE:")
        mostrar(MessageToDict(evento.message))
        textos_a2a.append(get_message_text(evento.message))

respuesta_a2a = "\n".join(t for t in textos_a2a if t.strip())
if not respuesta_a2a.strip():
    raise RuntimeError("No se recibió texto de respuesta por A2A.")
print("\nRESPUESTA A2A EN TEXTO:\n", respuesta_a2a)
evidencias["a2a"] = {"context_id": context_id_a2a, "pregunta": PREGUNTA_A2A, "respuesta": respuesta_a2a, "tasks": tareas_a2a}
if not tareas_a2a:
    print("REVISAR ENTREGA: hubo respuesta Message, pero esta ejecución no contiene la Task solicitada por el taller.")

MENSAJE A2A:
{
  "messageId": "020ea12f-df40-42c4-a19f-a9487f99d4f9",
  "contextId": "6c4673fd-37cc-473d-b8c4-0a35d5282b24",
  "role": "ROLE_USER",
  "parts": [
    {
      "text": "Usa el caso ficticio de tus apuntes para evaluar primero el impacto operacional y luego el retorno. Con evaluar_impacto_kpi mide los quiebres de stock: línea base 8 %, valor esperado 4 %, unidad %, mejor_si menor y sin valor_unitario_clp; la monetización se hace después con calcular_retorno. Después usa calcular_retorno con el costo del piloto (9.000.000 CLP) y el beneficio mensual del caso (1.200.000 CLP) a 12 meses. Consulta tus apuntes. Muestra la variación absoluta y porcentual del KPI, el beneficio acumulado, el ROI, los meses de recuperación y una conclusión breve. Declara los supuestos y recuerda que el caso es ficticio."
    }
  ]
}



TASK DEVUELTA:
{
  "id": "6c4673fd-37cc-473d-b8c4-0a35d5282b24:01a16a26-2af5-7129-92ca-d1e9bcfd9110",
  "contextId": "6c4673fd-37cc-473d-b8c4-0a35d5282b24",
  "status": {
    "state": "TASK_STATE_COMPLETED",
    "timestamp": "2026-10-23T19:04:14.313603Z"
  },
  "artifacts": [
    {
      "artifactId": "15764ccd-7343-4067-85f5-9bf711c22ffa",
      "name": "Assistant Response",
      "description": "Response from assistant 80c74e98-5251-5ce6-8713-2b1dd5835c0b",
      "parts": [
        {
          "text": "**Conclusión: el piloto es atractivo — bajar quiebres a la mitad y recuperar la inversión en 7,5 meses, con ROI 60 % a 12 meses.** (Caso ficticio de mis apuntes, no experiencia real.)\n\n**Impacto operacional (evaluar_impacto_kpi)**\n- Quiebres de stock: 8 % → 4 %; variación absoluta **-4 pp**, porcentual **-50 %**. Es mejora.\n\n**Retorno (calcular_retorno)**\n- Costo piloto: 9.000.000 CLP (dato del caso) · Beneficio mensual: 1.200.000 CLP\n- Beneficio acumulado 12 meses: **14.400.00

## 8. Revisar evidencias y completar tus observaciones

Esta revisión detecta secciones ejecutadas; **no evalúa automáticamente la calidad de las respuestas**. Contrasta el tono, datos, herramientas y memoria con el perfil diseñado.

In [11]:
requeridas = ["descubrimiento", "personalidad", "conocimientos", "memoria", "tarjeta", "a2a"]
for nombre in requeridas:
    print(f"{nombre}: {'EJECUTADO' if nombre in evidencias else 'PENDIENTE'}")
if "a2a" in evidencias:
    print("Task A2A guardada:", bool(evidencias["a2a"]["tasks"]))
print("Guarda el notebook con Ctrl+S para conservar sus salidas reales.")

descubrimiento: EJECUTADO
personalidad: EJECUTADO
conocimientos: EJECUTADO
memoria: EJECUTADO
tarjeta: EJECUTADO
a2a: EJECUTADO
Task A2A guardada: True
Guarda el notebook con Ctrl+S para conservar sus salidas reales.


### Párrafo para el documento de entrega · completar después de probar

**Estado:** pendiente de ejecución y revisión por Paulina.

- **Personalidad:** ¿Se presentó con tu nombre, especialidad y tono?
- **Conocimientos y habilidades:** ¿Consultó tus apuntes, usó `evaluar_impacto_kpi` y `calcular_retorno`? ¿Mostró la variación del KPI, el beneficio acumulado, ROI y payback del caso?
- **Memoria:** ¿Recordó el objetivo usando el mismo hilo?
- **A2A:** ¿Se descargó la tarjeta y volvió una Task completada con su resultado?
- **Ajustes:** indica qué cambiaste realmente en el prompt o apuntes después de probar. Si no cambiaste nada, explica por qué.

Redacta aquí el párrafo final con tus resultados. No presentes estas instrucciones como conclusiones ni des por exitosa una prueba que no corriste.

## Preparar mi tarjeta para la Etapa 3

En la integración del grupo, Paulina corre en el puerto **2027**, Rodrigo en el **2025** y Christopher en el **2026**. La celda actualiza el nombre y la descripción que el supervisor leerá en la Agent Card; es idempotente.

In [12]:
# Prepara la tarjeta de Paulina para la Etapa 3.
# El servidor A2A toma el nombre y la descripción del asistente para armar la Agent Card.
# Esta celda es idempotente: puedes ejecutarla más de una vez.
NOMBRE_TARJETA = "Agente Paulina"
DESCRIPCION_TARJETA = (
    "Especialista en estrategia y valor de negocio de IA y datos. Define el caso de negocio "
    "(problema, KPIs y beneficio esperado), evalúa el impacto de una iniciativa con el KPI que "
    "corresponda (mermas, tiempos de proceso, quiebres de stock, errores, horas ahorradas, "
    "satisfacción) y, cuando hay costo y beneficio monetario confiables, también su retorno. "
    "Propone un roadmap por fases partiendo por quick wins. No diseña arquitectura ni calcula "
    "costos de desarrollo; usa los aportes de otros especialistas como datos."
)

await lg.assistants.update(ASSISTANT_ID, name=NOMBRE_TARJETA, description=DESCRIPCION_TARJETA)
asistente_actualizado = await lg.assistants.get(ASSISTANT_ID)
print("Tarjeta preparada:")
print(json.dumps({
    "name": asistente_actualizado.get("name"),
    "description": asistente_actualizado.get("description"),
    "graph_id": asistente_actualizado.get("graph_id"),
    "assistant_id": asistente_actualizado.get("assistant_id"),
}, indent=2, ensure_ascii=False))

Tarjeta preparada:
{
  "name": "Agente Paulina",
  "description": "Especialista en estrategia y valor de negocio de IA y datos. Define el caso de negocio (problema, KPIs y beneficio esperado), evalúa el impacto de una iniciativa con el KPI que corresponda (mermas, tiempos de proceso, quiebres de stock, errores, horas ahorradas, satisfacción) y, cuando hay costo y beneficio monetario confiables, también su retorno. Propone un roadmap por fases partiendo por quick wins. No diseña arquitectura ni calcula costos de desarrollo; usa los aportes de otros especialistas como datos.",
  "graph_id": "agente_paulina",
  "assistant_id": "80c74e98-5251-5ce6-8713-2b1dd5835c0b"
}


## Etapa 3 · Conversación con Rodrigo y Christopher

La conversación entre los tres especialistas vive en el notebook compartido de la raíz:
[`notebooks/a2a-test.ipynb`](../../notebooks/a2a-test.ipynb).

## 9. Cerrar conexiones · ejecutar al terminar

No borra respuestas ni conversaciones del servidor. Si luego quieres repetir el envío A2A, ejecuta de nuevo la celda de tarjeta y cliente.

In [13]:
if "cliente_a2a" in globals():
    await cliente_a2a.close()
if "http_a2a" in globals() and not http_a2a.is_closed:
    await http_a2a.aclose()
print("Conexiones A2A cerradas. Guarda el notebook sin limpiar salidas.")

Conexiones A2A cerradas. Guarda el notebook sin limpiar salidas.


## Si aparece un error

- **No conecta al puerto 2027:** revisa que la terminal del servidor esté abierta, dentro de `agente-paulina` y sin errores de carga.
- **No aparece agente_paulina:** verifica el listado y que ejecutaste el servidor de esta copia.
- **Clave rechazada, modelo inexistente o cuota agotada:** revisa la configuración/cuenta del proveedor en la terminal; no pegues claves en celdas ni capturas.
- **Timeout:** revisa la terminal antes de reintentar: la tarea anterior podría continuar. TIMEOUT está en la primera celda.